# Human Proteome Annotation Ceiling

This notebook compares the seven bundled UdonPred evaluation datasets against the local human proteome FASTA. The human proteome file contains sequences but no residue-level disorder annotations, so the result is an overlap/coverage ceiling: it shows how much of each annotated UdonPred test set can be placed on exact human-proteome sequences.

Outputs are written to `results/human_proteome_annotation_ceiling/`:

- `human_proteome_overlap_summary.csv`
- `human_proteome_overlap_details.csv`
- `human_proteome_overlap_summary.json`
- `human_proteome_overlap_counts.png`
- `human_proteome_overlap_fraction.png`

In [ ]:
from __future__ import annotations

import json
import re
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "UdonPred").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "scripts"))
from run_simple_baselines import DATASETS, read_jsonl, valid_mask

UDONPRED_DIR = REPO_ROOT / "UdonPred"
HUMAN_PROTEOME_FASTA = REPO_ROOT / "HumanProteome" / "human_preteome.fasta"
OUTPUT_DIR = REPO_ROOT / "results" / "human_proteome_annotation_ceiling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS

## Load Human Proteome

The FASTA headers are parsed for UniProt-style accessions such as `sp|P12345|NAME_HUMAN`. Exact sequence matching is the primary comparison because most UdonPred records do not use UniProt accessions as their IDs.

In [ ]:
@dataclass(frozen=True)
class FastaRecord:
    header: str
    sequence: str
    accession: str
    gene: str


def parse_accession(header: str) -> str:
    parts = header.split()
    first = parts[0]
    pipe_parts = first.split("|")
    if len(pipe_parts) >= 2 and pipe_parts[1]:
        return pipe_parts[1]
    return first


def parse_gene(header: str) -> str:
    match = re.search(r"\bGN=([^\s]+)", header)
    return match.group(1) if match else ""


def read_fasta(path: Path) -> list[FastaRecord]:
    records = []
    header = None
    chunks = []
    with path.open() as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    sequence = "".join(chunks)
                    records.append(FastaRecord(header, sequence, parse_accession(header), parse_gene(header)))
                header = line[1:]
                chunks = []
            else:
                chunks.append(line)
    if header is not None:
        sequence = "".join(chunks)
        records.append(FastaRecord(header, sequence, parse_accession(header), parse_gene(header)))
    return records


human_records = read_fasta(HUMAN_PROTEOME_FASTA)
human_by_sequence = defaultdict(list)
human_by_accession = defaultdict(list)
human_by_gene = defaultdict(list)
for record in human_records:
    human_by_sequence[record.sequence].append(record)
    human_by_accession[record.accession].append(record)
    if record.gene:
        human_by_gene[record.gene].append(record)

human_stats = {
    "n_human_proteome_records": len(human_records),
    "n_unique_human_sequences": len(human_by_sequence),
    "n_unique_human_accessions": len(human_by_accession),
    "human_residues_total": sum(len(record.sequence) for record in human_records),
}
human_stats

## Compare UdonPred Datasets

For each UdonPred test record, the notebook checks:

1. exact sequence match against the human proteome,
2. exact accession match if the UdonPred ID looks like a UniProt accession,
3. exact gene-name match as a secondary diagnostic.

Residue counts use the UdonPred valid-label mask, so masked residues do not inflate annotated coverage.

In [ ]:
def first_or_empty(values: list[str]) -> str:
    return values[0] if values else ""


def match_modes(protein_id: str, sequence: str) -> tuple[list[str], list[FastaRecord]]:
    matches_by_key = {}
    modes_by_key = defaultdict(list)

    for mode, records in (
        ("sequence", human_by_sequence.get(sequence, [])),
        ("accession", human_by_accession.get(protein_id, [])),
        ("gene", human_by_gene.get(protein_id, [])),
    ):
        for record in records:
            key = (record.accession, record.header)
            matches_by_key[key] = record
            modes_by_key[key].append(mode)

    modes = sorted({mode for mode_list in modes_by_key.values() for mode in mode_list})
    return modes, list(matches_by_key.values())


summary_rows = []
detail_rows = []

for dataset in DATASETS:
    records = read_jsonl(UDONPRED_DIR / "data" / dataset / "test.jsonl")
    total_records = len(records)
    total_residues = sum(len(record.sequence) for record in records)
    total_valid_residues = sum(int(valid_mask(record.labels).sum()) for record in records)
    matched_records = 0
    matched_residues = 0
    matched_valid_residues = 0
    sequence_matches = 0
    accession_matches = 0
    gene_matches = 0

    for record in records:
        modes, human_matches = match_modes(record.protein_id, record.sequence)
        if not modes:
            continue

        valid_residues = int(valid_mask(record.labels).sum())
        matched_records += 1
        matched_residues += len(record.sequence)
        matched_valid_residues += valid_residues
        sequence_matches += int("sequence" in modes)
        accession_matches += int("accession" in modes)
        gene_matches += int("gene" in modes)

        detail_rows.append(
            {
                "dataset": dataset,
                "udonpred_id": record.protein_id,
                "sequence_length": len(record.sequence),
                "valid_annotated_residues": valid_residues,
                "match_mode": "+".join(modes),
                "n_human_matches": len(human_matches),
                "human_accessions": ";".join(sorted({match.accession for match in human_matches})),
                "human_genes": ";".join(sorted({match.gene for match in human_matches if match.gene})),
                "example_human_header": first_or_empty([match.header for match in human_matches]),
            }
        )

    summary_rows.append(
        {
            "dataset": dataset,
            "n_udonpred_test_proteins": total_records,
            "n_matched_human_proteins": matched_records,
            "protein_overlap_fraction": matched_records / total_records if total_records else np.nan,
            "n_udonpred_test_residues": total_residues,
            "n_matched_human_residues": matched_residues,
            "residue_overlap_fraction": matched_residues / total_residues if total_residues else np.nan,
            "n_valid_annotated_residues": total_valid_residues,
            "n_matched_valid_annotated_residues": matched_valid_residues,
            "valid_annotated_residue_overlap_fraction": matched_valid_residues / total_valid_residues if total_valid_residues else np.nan,
            "n_sequence_matches": sequence_matches,
            "n_accession_matches": accession_matches,
            "n_gene_matches": gene_matches,
            "notes": "human proteome FASTA has no residue-level labels; exact sequence match is the primary comparison",
        }
    )

summary_df = pd.DataFrame(summary_rows)
detail_df = pd.DataFrame(detail_rows)
summary_df

In [ ]:
summary_path = OUTPUT_DIR / "human_proteome_overlap_summary.csv"
detail_path = OUTPUT_DIR / "human_proteome_overlap_details.csv"
json_path = OUTPUT_DIR / "human_proteome_overlap_summary.json"

summary_df.to_csv(summary_path, index=False)
detail_df.to_csv(detail_path, index=False)
with json_path.open("w") as handle:
    json.dump(
        {
            "human_proteome": human_stats,
            "datasets": summary_df.to_dict(orient="records"),
        },
        handle,
        indent=2,
    )

summary_path, detail_path, json_path

## Visualize Coverage

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
plot_df = summary_df.sort_values("n_matched_human_proteins", ascending=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(plot_df["dataset"], plot_df["n_udonpred_test_proteins"], label="UdonPred test proteins", color="#d9dee8")
ax.bar(plot_df["dataset"], plot_df["n_matched_human_proteins"], label="Matched human proteins", color="#2f6f9f")
ax.set_ylabel("Proteins")
ax.set_xlabel("Dataset")
ax.set_title("UdonPred test proteins with exact human-proteome matches")
ax.legend(frameon=False)
fig.tight_layout()
counts_path = OUTPUT_DIR / "human_proteome_overlap_counts.png"
fig.savefig(counts_path, dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
fraction_df = summary_df.sort_values("valid_annotated_residue_overlap_fraction", ascending=False)
ax.bar(
    fraction_df["dataset"],
    fraction_df["valid_annotated_residue_overlap_fraction"],
    color="#3d8f73",
)
ax.set_ylim(0, 1)
ax.set_ylabel("Fraction of valid annotated residues")
ax.set_xlabel("Dataset")
ax.set_title("Annotated UdonPred residues covered by exact human-proteome matches")
fig.tight_layout()
fraction_path = OUTPUT_DIR / "human_proteome_overlap_fraction.png"
fig.savefig(fraction_path, dpi=200)
plt.show()

counts_path, fraction_path

## Matched Proteins

Use the detail table to inspect which proteins drive the overlap. Multiple human proteome accessions may share an exact sequence.

In [ ]:
detail_df.sort_values(["dataset", "sequence_length"], ascending=[True, False]).head(25)